In [0]:
%pip install yfinance

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import yfinance as yf
import pandas as pd
from datetime import datetime

acoes = {
    "ITUB4.SA": "bancos",
    "BBDC4.SA": "bancos",
    "BBAS3.SA": "bancos",
    "VALE3.SA": "commodities",
    "PETR4.SA": "commodities",
    "BRAP4.SA": "commodities",
    "MGLU3.SA": "varejo",
    "LREN3.SA": "varejo",
    "TOTS3.SA": "tech",
    "VIVT3.SA": "telecom"
}

frames = []
for ticker, setor in acoes.items():
    print(f"Coletando {ticker}...")
    df = yf.download(
        ticker,
        start="2022-01-01",
        end=datetime.today().strftime('%Y-%m-%d'),
        progress=False,
        auto_adjust=True
    )
    df.columns = [col[0].lower() if isinstance(col, tuple) else col.lower() 
                  for col in df.columns]
    df["ticker"] = ticker.replace(".SA", "")
    df["setor"]  = setor
    frames.append(df)

df_all = pd.concat(frames).reset_index()
df_all.columns = [col[0].lower() if isinstance(col, tuple) else col.lower() 
                  for col in df_all.columns]

print(f"\nTotal de registros: {len(df_all)}")
print(df_all.head())

Coletando ITUB4.SA...
Coletando BBDC4.SA...
Coletando BBAS3.SA...
Coletando VALE3.SA...
Coletando PETR4.SA...
Coletando BRAP4.SA...
Coletando MGLU3.SA...
Coletando LREN3.SA...
Coletando TOTS3.SA...
Coletando VIVT3.SA...

Total de registros: 11040
        date      close       high  ...    volume  ticker   setor
0 2022-01-03  14.157307  14.354759  ...  37222335   ITUB4  bancos
1 2022-01-04  14.558789  14.598279  ...  49793763   ITUB4  bancos
2 2022-01-05  14.282354  14.624605  ...  35851405   ITUB4  bancos
3 2022-01-06  14.571957  14.637774  ...  40624734   ITUB4  bancos
4 2022-01-07  14.894464  14.907627  ...  48590177   ITUB4  bancos

[5 rows x 8 columns]


In [0]:
df_spark = spark.createDataFrame(df_all)

spark.sql("CREATE DATABASE IF NOT EXISTS b3_pipeline")

df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.bronze_b3_stocks")

print("✅ Bronze salvo com sucesso!")
df_spark.printSchema()

✅ Bronze salvo com sucesso!
root
 |-- date: timestamp (nullable = true)
 |-- close: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- open: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- ticker: string (nullable = true)
 |-- setor: string (nullable = true)

